# Black Hole Gravity — Vertical Shorts Video Notebook

This notebook generates a **vertical YouTube Shorts-style explainer video** showing **how gravity works near black holes**.

It creates a procedural/stylized animation with:

- curved spacetime grid / gravity well visualization
- falling objects accelerating inward
- event horizon and escape limits
- light bending / gravitational lensing
- accretion disk glow and tidal stretching
- cinematic captions, progress bar, vignette, starfield, and MP4 export

**Outputs**

- `Black_Hole_Gravity_Explained_Shorts.mp4`
- `Black_Hole_Gravity_Storyboard.png`

> Note: This is an educational visualization, not a physically exact general-relativity ray tracer. It is designed to explain the concept visually in a Shorts/reels format. You can tune the look, speed, script, and resolution in the settings cell.

In [ ]:
# In a fresh Colab/Jupyter environment, uncomment this line:
# %pip install -U numpy pillow imageio imageio-ffmpeg tqdm

## Source notes for concepts used in captions

- In general relativity, gravity can be described as the curvature of spacetime.
- A black hole has an **event horizon**: a boundary beyond which light cannot escape.
- Near a black hole, light paths bend strongly, producing gravitational lensing.
- Matter outside the event horizon can orbit, heat up, and form a bright accretion disk.
- Strong tidal gravity can stretch objects because gravity is much stronger on the side closer to the black hole.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import random
import textwrap
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import imageio.v2 as imageio

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

# ==========================
# Render settings
# ==========================
OUT_DIR = Path("black_hole_gravity_output")
OUT_DIR.mkdir(exist_ok=True)

DRAFT_MODE = True  # True = faster preview. False = sharper final export.

if DRAFT_MODE:
    W, H = 576, 1024   # exact 9:16 and codec-friendly
    FPS = 20
else:
    W, H = 720, 1280   # exact 9:16
    FPS = 24

VIDEO_SECONDS = 46.0
NFRAMES = int(VIDEO_SECONDS * FPS)

VIDEO_NAME = OUT_DIR / "Black_Hole_Gravity_Explained_Shorts.mp4"
STORYBOARD_NAME = OUT_DIR / "Black_Hole_Gravity_Storyboard.png"

SEED = 87
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Main black hole location in screen coordinates
BH_CENTER = (W * 0.50, H * 0.48)
BH_RADIUS = int(W * 0.105)        # visual event-horizon radius
PHOTON_RING_RADIUS = int(W * 0.145)

SCENES = [
    {
        "start": 0.0,
        "end": 6.0,
        "kind": "hook",
        "eyebrow": "BLACK HOLE GRAVITY",
        "title": "SPACE\nBENDS HERE",
        "caption": "A black hole does not just pull objects — it warps the paths they can follow.",
    },
    {
        "start": 6.0,
        "end": 14.0,
        "kind": "curvature",
        "eyebrow": "1  CURVED SPACETIME",
        "title": "Gravity is\ngeometry",
        "caption": "Mass curves spacetime. Near a black hole, the curve becomes extreme.",
    },
    {
        "start": 14.0,
        "end": 22.0,
        "kind": "falling",
        "eyebrow": "2  FALLING IN",
        "title": "Paths tilt\ninward",
        "caption": "Objects still follow the straightest possible paths — but spacetime itself is bent.",
    },
    {
        "start": 22.0,
        "end": 30.0,
        "kind": "horizon",
        "eyebrow": "3  EVENT HORIZON",
        "title": "No escape\ninside",
        "caption": "At the event horizon, every future path points deeper inward — even for light.",
    },
    {
        "start": 30.0,
        "end": 38.0,
        "kind": "lensing",
        "eyebrow": "4  LIGHT BENDS",
        "title": "Gravity\nlenses light",
        "caption": "Light from behind can curve around the black hole, forming arcs and rings.",
    },
    {
        "start": 38.0,
        "end": 46.0,
        "kind": "outro",
        "eyebrow": "THE IDEA",
        "title": "The closer you get,\nthe less escape\nremains",
        "caption": "Black holes reveal gravity at its strongest: spacetime guiding motion, light, and time.",
    },
]

In [ ]:
# ==========================
# Fonts, colors, easing, text
# ==========================
def font_path(bold: bool = False) -> str | None:
    candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf",
    ]
    for p in candidates:
        if p and Path(p).exists():
            return p
    return None

def load_font(size: int, bold: bool = False):
    p = font_path(bold)
    if p:
        return ImageFont.truetype(p, size)
    return ImageFont.load_default()

FONT_EYEBROW = load_font(max(16, W // 34), True)
FONT_TITLE = load_font(max(40, W // 9), True)
FONT_SUBTITLE = load_font(max(28, W // 15), True)
FONT_BODY = load_font(max(22, W // 24), False)
FONT_SMALL = load_font(max(15, W // 38), False)
FONT_LABEL = load_font(max(18, W // 31), True)

ICE = (182, 226, 255, 255)
WHITE = (245, 250, 255, 255)
GOLD = (255, 190, 82, 255)
ORANGE = (255, 107, 48, 255)
MAGENTA = (220, 120, 255, 255)
CYAN = (90, 220, 255, 255)
GRID = (80, 150, 210, 135)
DARK = (2, 3, 10, 255)


def clamp01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))


def ease_in_out(x: float) -> float:
    x = clamp01(x)
    return x * x * (3 - 2 * x)


def ease_out(x: float) -> float:
    x = clamp01(x)
    return 1 - (1 - x) ** 3


def ease_in(x: float) -> float:
    x = clamp01(x)
    return x ** 3


def lerp(a: float, b: float, x: float) -> float:
    return a + (b - a) * x


def draw_text_glow(
    img: Image.Image,
    xy,
    text: str,
    font,
    fill=(255, 255, 255, 255),
    glow=(80, 160, 255, 130),
    blur=8,
    anchor=None,
    stroke_width=0,
    stroke_fill=(0, 0, 0, 180),
):
    layer = Image.new("RGBA", img.size, (0, 0, 0, 0))
    ld = ImageDraw.Draw(layer, "RGBA")
    ld.multiline_text(
        xy,
        text,
        font=font,
        fill=glow,
        anchor=anchor,
        align="center",
        spacing=int(W * 0.012),
        stroke_width=stroke_width,
        stroke_fill=stroke_fill,
    )
    layer = layer.filter(ImageFilter.GaussianBlur(blur))
    img.alpha_composite(layer)
    d = ImageDraw.Draw(img, "RGBA")
    d.multiline_text(
        xy,
        text,
        font=font,
        fill=fill,
        anchor=anchor,
        align="center",
        spacing=int(W * 0.012),
        stroke_width=stroke_width,
        stroke_fill=stroke_fill,
    )


def wrap_text(text: str, width: int = 28) -> str:
    return "\n".join(textwrap.wrap(text, width=width))


def rounded_rectangle_alpha(size, radius, fill):
    im = Image.new("RGBA", size, (0, 0, 0, 0))
    d = ImageDraw.Draw(im, "RGBA")
    d.rounded_rectangle([0, 0, size[0] - 1, size[1] - 1], radius=radius, fill=fill)
    return im


def draw_caption_box(img: Image.Image, text: str, y: int | None = None):
    margin = int(W * 0.07)
    if y is None:
        y = int(H * 0.775)
    body = wrap_text(text, 34 if W <= 600 else 40)
    lines = body.count("\n") + 1
    box_h = int(H * 0.105) + (lines - 1) * int(H * 0.030)
    box = rounded_rectangle_alpha((W - 2 * margin, box_h), int(W * 0.045), (4, 8, 20, 185))
    glow = rounded_rectangle_alpha((W - 2 * margin, box_h), int(W * 0.045), (70, 170, 255, 35)).filter(ImageFilter.GaussianBlur(10))
    img.alpha_composite(glow, (margin, y))
    img.alpha_composite(box, (margin, y))
    d = ImageDraw.Draw(img, "RGBA")
    d.multiline_text((margin + int(W * 0.04), y + int(H * 0.028)), body, font=FONT_BODY, fill=WHITE, spacing=7)


def dark_gradient_overlay(img: Image.Image, top_alpha=120, bottom_alpha=190):
    arr = np.zeros((H, W, 4), dtype=np.uint8)
    y = np.linspace(0, 1, H)
    alpha = (top_alpha * (1 - y) + bottom_alpha * y).astype(np.uint8)
    arr[..., 3] = alpha[:, None]
    overlay = Image.fromarray(arr, "RGBA")
    img.alpha_composite(overlay)


def vignette(img: Image.Image, strength=0.62) -> Image.Image:
    yy, xx = np.mgrid[0:H, 0:W]
    cx, cy = W / 2, H / 2
    r = np.sqrt(((xx - cx) / (W * 0.67)) ** 2 + ((yy - cy) / (H * 0.60)) ** 2)
    mask = np.clip((r - 0.08) / 0.92, 0, 1) ** 1.55
    alpha = (mask * 255 * strength).astype(np.uint8)
    overlay = np.zeros((H, W, 4), dtype=np.uint8)
    overlay[..., 3] = alpha
    out = img.copy()
    out.alpha_composite(Image.fromarray(overlay, "RGBA"))
    return out


def draw_progress_bar(img: Image.Image, t: float):
    d = ImageDraw.Draw(img, "RGBA")
    bar_h = max(4, H // 180)
    x0 = int(W * 0.08)
    x1 = int(W * 0.92)
    y = int(H * 0.965)
    d.rounded_rectangle([x0, y, x1, y + bar_h], radius=bar_h, fill=(255, 255, 255, 42))
    p = clamp01(t / VIDEO_SECONDS)
    d.rounded_rectangle([x0, y, int(x0 + (x1 - x0) * p), y + bar_h], radius=bar_h, fill=(255, 174, 80, 220))


def draw_header(img: Image.Image, scene: dict, local: float, warm=False):
    margin = int(W * 0.07)
    y = int(H * 0.065)
    col = GOLD if warm else ICE
    glow = (255, 130, 45, 120) if warm else (60, 145, 255, 125)
    draw_text_glow(img, (margin, y), scene["eyebrow"], FONT_EYEBROW, fill=col, glow=glow, blur=6)
    d = ImageDraw.Draw(img, "RGBA")
    line_y = y + int(H * 0.035)
    d.rounded_rectangle([margin, line_y, margin + int(W * (0.25 + 0.25 * ease_out(local))), line_y + 3], radius=2, fill=col)


def fit_title_font(text: str, max_width: int | None = None, start_size: int | None = None, min_size: int = 30):
    """Choose a title font that fits the vertical-video width."""
    if max_width is None:
        max_width = int(W * 0.88)
    if start_size is None:
        start_size = max(40, W // 9)
    test_img = Image.new("RGB", (10, 10))
    td = ImageDraw.Draw(test_img)
    lines = text.split("\n")
    for size in range(start_size, min_size - 1, -2):
        font = load_font(size, True)
        widths = []
        for line in lines:
            bbox = td.textbbox((0, 0), line, font=font, stroke_width=2)
            widths.append(bbox[2] - bbox[0])
        if max(widths or [0]) <= max_width:
            return font
    return load_font(min_size, True)


def draw_big_title(img: Image.Image, text: str, y: int, warm: bool = True):
    title_font = fit_title_font(text)
    draw_text_glow(
        img,
        (W // 2, y),
        text,
        title_font,
        fill=(255, 246, 215, 255) if warm else WHITE,
        glow=(255, 110, 40, 155) if warm else (75, 160, 255, 135),
        anchor="mm",
        blur=12,
        stroke_width=2,
        stroke_fill=(0, 0, 0, 230),
    )


In [ ]:
# ==========================
# Procedural background assets
# ==========================
STAR_COUNT = 1000 if DRAFT_MODE else 1500
STAR_X = rng.random(STAR_COUNT) * W
STAR_Y = rng.random(STAR_COUNT) * H
STAR_SIZE = rng.random(STAR_COUNT) * 1.7 + 0.25
STAR_PHASE = rng.random(STAR_COUNT) * math.tau
STAR_COLOR_PICK = rng.integers(0, 4, STAR_COUNT)
STAR_COLORS = np.array([
    [210, 230, 255],
    [255, 255, 240],
    [255, 214, 160],
    [160, 205, 255],
], dtype=np.uint8)


def draw_space(img: Image.Image, t: float, tint=(2, 3, 12)):
    d = ImageDraw.Draw(img, "RGBA")
    d.rectangle([0, 0, W, H], fill=(tint[0], tint[1], tint[2], 255))

    # Subtle colored nebula haze
    neb = Image.new("RGBA", img.size, (0, 0, 0, 0))
    nd = ImageDraw.Draw(neb, "RGBA")
    for k in range(7):
        cx = int((0.12 + 0.80 * ((k * 0.381) % 1)) * W + math.sin(t * 0.08 + k) * 22)
        cy = int((0.10 + 0.82 * ((k * 0.613) % 1)) * H + math.cos(t * 0.06 + k) * 28)
        r = int(W * (0.20 + 0.08 * ((k * 0.47) % 1)))
        color = [(30, 80, 170, 22), (90, 40, 150, 20), (190, 72, 28, 16)][k % 3]
        nd.ellipse([cx - r, cy - r, cx + r, cy + r], fill=color)
    neb = neb.filter(ImageFilter.GaussianBlur(int(W * 0.08)))
    img.alpha_composite(neb)

    # Star field with tiny parallax drift
    for i in range(STAR_COUNT):
        x = (STAR_X[i] + t * (0.7 + STAR_SIZE[i] * 0.25)) % W
        y = (STAR_Y[i] + math.sin(t * 0.15 + STAR_PHASE[i]) * 0.8) % H
        twinkle = 0.60 + 0.40 * math.sin(t * 2.2 + STAR_PHASE[i]) ** 2
        col = STAR_COLORS[STAR_COLOR_PICK[i]]
        a = int(65 + 160 * twinkle)
        s = STAR_SIZE[i]
        d.ellipse([x - s, y - s, x + s, y + s], fill=(int(col[0]), int(col[1]), int(col[2]), a))


def make_accretion_disk_asset(size: int = 920) -> Image.Image:
    """A reusable tilted glowing disk that can be rotated/blurred/placed."""
    asset = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    d = ImageDraw.Draw(asset, "RGBA")
    cx = cy = size // 2

    for i in range(36):
        frac = i / 35
        w = int(size * (0.020 + 0.018 * (1 - frac)))
        rx = int(size * (0.24 + 0.30 * frac))
        ry = int(size * (0.050 + 0.092 * frac))
        hue = frac
        if frac < 0.38:
            col = (255, int(235 - 90 * frac), int(120 - 90 * frac), int(85 * (1 - frac * 0.2)))
        else:
            col = (255, int(110 - 40 * min(frac, 1)), int(35 + 15 * frac), int(105 * (1 - 0.45 * frac)))
        bbox = [cx - rx, cy - ry, cx + rx, cy + ry]
        start = -178 + int(20 * math.sin(frac * math.tau))
        end = 178 - int(10 * math.cos(frac * math.tau))
        d.arc(bbox, start=start, end=end, fill=col, width=max(2, w))

    # Hot asymmetric streaks for orbital motion
    for k in range(20):
        a0 = -165 + k * 17
        a1 = a0 + 18 + (k % 4) * 4
        rx = int(size * (0.32 + 0.20 * ((k * 0.37) % 1)))
        ry = int(size * (0.075 + 0.045 * ((k * 0.53) % 1)))
        col = (255, 205, 85, 80) if k % 3 else (255, 90, 50, 90)
        d.arc([cx - rx, cy - ry, cx + rx, cy + ry], start=a0, end=a1, fill=col, width=int(size * 0.016))

    asset = asset.filter(ImageFilter.GaussianBlur(0.35))
    return asset

ACCRETION_ASSET = make_accretion_disk_asset(960)


def draw_black_hole(img: Image.Image, t: float, cx: float | None = None, cy: float | None = None, scale=1.0, disk=True, ring=True):
    if cx is None:
        cx = BH_CENTER[0]
    if cy is None:
        cy = BH_CENTER[1]

    # Place rotating accretion disk behind the shadow
    if disk:
        disk_img = ACCRETION_ASSET.rotate(math.degrees(0.10 * math.sin(t * 0.35)), resample=Image.Resampling.BICUBIC)
        ds = int(W * 1.35 * scale)
        disk_img = disk_img.resize((ds, ds), Image.Resampling.LANCZOS)
        img.alpha_composite(disk_img, (int(cx - ds / 2), int(cy - ds / 2)))

    d = ImageDraw.Draw(img, "RGBA")

    # Photon ring glow
    if ring:
        glow = Image.new("RGBA", img.size, (0, 0, 0, 0))
        gd = ImageDraw.Draw(glow, "RGBA")
        for j in range(10):
            r = int((PHOTON_RING_RADIUS + j * 4) * scale)
            a = int(58 * (1 - j / 10))
            gd.ellipse([cx - r, cy - r, cx + r, cy + r], outline=(255, 174, 72, a), width=max(2, int(3 * scale)))
        glow = glow.filter(ImageFilter.GaussianBlur(max(2, int(4 * scale))))
        img.alpha_composite(glow)
        r = int(PHOTON_RING_RADIUS * scale)
        d.ellipse([cx - r, cy - r, cx + r, cy + r], outline=(255, 200, 105, 185), width=max(2, int(3 * scale)))

    # Black shadow + inner darkness
    for j in range(8, -1, -1):
        r = int((BH_RADIUS + j * 9) * scale)
        a = int(52 * (1 - j / 9))
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(0, 0, 0, a))
    r = int(BH_RADIUS * scale)
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(0, 0, 0, 255))

    # Event horizon label ring
    d.ellipse([cx - r, cy - r, cx + r, cy + r], outline=(50, 80, 120, 210), width=max(1, int(2 * scale)))


def draw_lens_distorted_stars(img: Image.Image, t: float, strength=1.0):
    """Decorative arcs around the black hole to suggest gravitational lensing."""
    d = ImageDraw.Draw(img, "RGBA")
    cx, cy = BH_CENTER
    for k in range(34):
        base = (k * 0.61803398875) % 1
        rr = W * (0.18 + 0.52 * base)
        angle = k * 0.91 + t * 0.07
        stretch = 10 + 34 * (1 - base) * strength
        x = cx + math.cos(angle) * rr
        y = cy + math.sin(angle) * rr * 0.78
        col = (170, 220, 255, int(55 + 80 * (1 - base)))
        d.arc([x-stretch, y-stretch*0.45, x+stretch, y+stretch*0.45], start=10, end=170, fill=col, width=1)

In [ ]:
# ==========================
# Physics-inspired drawing helpers
# ==========================
def bend_point(x: float, y: float, cx: float, cy: float, strength: float = 0.55):
    """Stylized screen-space bending toward the black hole. Not a real GR transform."""
    dx = x - cx
    dy = y - cy
    r2 = dx * dx + dy * dy
    r = math.sqrt(r2) + 1e-6
    # Stronger bending near the black hole, smoothly fading outward
    influence = strength * (W * 0.34) ** 2 / (r2 + (W * 0.22) ** 2)
    pull = min(0.62, influence)
    # Also compress vertically to look like a gravity well on a flat sheet
    return x - dx * pull, y - dy * pull + (W * 0.020) * pull * math.sin(dx / W * math.tau)


def draw_curved_grid(img: Image.Image, t: float, local: float, strength=0.58, alpha=125):
    d = ImageDraw.Draw(img, "RGBA")
    cx, cy = BH_CENTER
    spacing = int(W * 0.105)
    extent = int(W * 0.95)

    # Horizontal grid lines, warped inward
    for y0 in range(int(cy - extent), int(cy + extent), spacing):
        pts = []
        for x in np.linspace(-W * 0.12, W * 1.12, 100):
            bx, by = bend_point(x, y0, cx, cy, strength * ease_out(local))
            by += math.sin(t * 0.6 + x * 0.012) * 1.2
            pts.append((bx, by))
        d.line(pts, fill=(80, 170, 235, alpha), width=1)

    # Vertical grid lines, warped inward
    for x0 in range(int(cx - extent), int(cx + extent), spacing):
        pts = []
        for y in np.linspace(-H * 0.04, H * 0.97, 120):
            bx, by = bend_point(x0, y, cx, cy, strength * ease_out(local))
            bx += math.sin(t * 0.5 + y * 0.010) * 1.2
            pts.append((bx, by))
        d.line(pts, fill=(70, 140, 220, int(alpha * 0.78)), width=1)


def draw_gravity_vectors(img: Image.Image, t: float, local: float):
    d = ImageDraw.Draw(img, "RGBA")
    cx, cy = BH_CENTER
    for gy in np.linspace(H * 0.20, H * 0.68, 5):
        for gx in np.linspace(W * 0.18, W * 0.82, 5):
            dx = cx - gx
            dy = cy - gy
            dist = math.hypot(dx, dy) + 1e-5
            if dist < BH_RADIUS * 1.7:
                continue
            ux, uy = dx / dist, dy / dist
            mag = W * (0.026 + 0.06 * min(1.0, (W * 0.34 / dist) ** 1.8)) * ease_out(local)
            x2, y2 = gx + ux * mag, gy + uy * mag
            a = int(55 + 115 * min(1, (W * 0.34 / dist) ** 1.4))
            d.line([gx, gy, x2, y2], fill=(255, 185, 80, a), width=2)
            # arrow head
            ang = math.atan2(uy, ux)
            for off in (2.55, -2.55):
                hx = x2 + math.cos(ang + off) * W * 0.015
                hy = y2 + math.sin(ang + off) * W * 0.015
                d.line([x2, y2, hx, hy], fill=(255, 185, 80, a), width=2)


def draw_orbit_path(img: Image.Image, t: float, local: float):
    d = ImageDraw.Draw(img, "RGBA")
    cx, cy = BH_CENTER
    # Stable-looking outer orbit
    d.ellipse([cx-W*0.34, cy-W*0.15, cx+W*0.34, cy+W*0.15], outline=(120, 210, 255, int(95*local)), width=2)
    # Falling spiral path
    pts = []
    for i in range(200):
        u = i / 199
        ang = -2.4 * math.tau * u + t * 0.22
        rr = W * lerp(0.45, 0.13, u)
        x = cx + math.cos(ang) * rr
        y = cy + math.sin(ang) * rr * 0.46
        pts.append((x, y))
    d.line(pts, fill=(255, 190, 90, int(150 * local)), width=3)

    # Small falling object moving on spiral
    u = ease_in_out(local)
    ang = -2.4 * math.tau * u + t * 0.22
    rr = W * lerp(0.45, 0.13, u)
    x = cx + math.cos(ang) * rr
    y = cy + math.sin(ang) * rr * 0.46
    glow = Image.new("RGBA", img.size, (0, 0, 0, 0))
    gd = ImageDraw.Draw(glow, "RGBA")
    gd.ellipse([x-18, y-18, x+18, y+18], fill=(255, 210, 120, 90))
    glow = glow.filter(ImageFilter.GaussianBlur(8))
    img.alpha_composite(glow)
    d.ellipse([x-6, y-6, x+6, y+6], fill=(255, 248, 210, 255))


def draw_event_horizon_cones(img: Image.Image, t: float, local: float):
    d = ImageDraw.Draw(img, "RGBA")
    cx, cy = BH_CENTER
    zones = [
        (cx - W * 0.30, cy - W * 0.02, -0.55, "outside"),
        (cx - W * 0.05, cy + W * 0.21, -1.30, "near"),
        (cx + W * 0.16, cy + W * 0.04, -2.05, "inside"),
    ]
    for x, y, tilt, label in zones:
        amount = ease_out(local)
        ang = tilt * amount
        length = W * 0.105
        spread = 0.55
        p1 = (x, y)
        p2 = (x + math.cos(ang-spread)*length, y + math.sin(ang-spread)*length)
        p3 = (x + math.cos(ang+spread)*length, y + math.sin(ang+spread)*length)
        fill = (90, 210, 255, 34) if label != "inside" else (255, 100, 70, 42)
        line = (110, 225, 255, 160) if label != "inside" else (255, 130, 80, 180)
        d.polygon([p1, p2, p3], fill=fill)
        d.line([p1, p2], fill=line, width=2)
        d.line([p1, p3], fill=line, width=2)
        if label == "inside":
            d.text((x - W * 0.08, y + W * 0.11), "future paths\npoint inward", font=FONT_SMALL, fill=(255, 190, 150, 210), align="center")


def draw_light_ray(img: Image.Image, y0: float, offset: float, t: float, color, width=3):
    d = ImageDraw.Draw(img, "RGBA")
    cx, cy = BH_CENTER
    pts = []
    for x in np.linspace(-W * 0.10, W * 1.10, 220):
        dx = x - cx
        # Bending formula creates an S-curve around the black hole
        bend = offset * (W * 0.32) ** 2 / (dx * dx + (W * 0.17) ** 2)
        y = y0 + bend * math.copysign(1, y0 - cy)
        if math.hypot(x - cx, y - cy) < BH_RADIUS * 1.06:
            continue
        pts.append((x, y))
    if len(pts) > 1:
        d.line(pts, fill=color, width=width)


def draw_tidal_object(img: Image.Image, t: float, local: float):
    """Stretches a small object as it approaches the black hole."""
    d = ImageDraw.Draw(img, "RGBA")
    cx, cy = BH_CENTER
    u = ease_in_out(local)
    x = lerp(W * 0.22, cx - W * 0.13, u)
    y = lerp(H * 0.36, cy + W * 0.08, u)
    stretch = lerp(1.0, 3.8, u)
    angle = math.atan2(cy - y, cx - x)
    length = W * 0.055 * stretch
    width = W * 0.026 / (0.7 + stretch * 0.22)
    # draw rotated capsule as line + endpoints
    x1 = x - math.cos(angle) * length / 2
    y1 = y - math.sin(angle) * length / 2
    x2 = x + math.cos(angle) * length / 2
    y2 = y + math.sin(angle) * length / 2
    d.line([x1, y1, x2, y2], fill=(255, 210, 140, 230), width=max(4, int(width)))
    r = max(2, int(width * 0.52))
    d.ellipse([x1-r, y1-r, x1+r, y1+r], fill=(255, 238, 190, 230))
    d.ellipse([x2-r, y2-r, x2+r, y2+r], fill=(255, 150, 90, 230))
    d.text((W * 0.08, H * 0.63), "tidal stretching", font=FONT_LABEL, fill=(255, 200, 130, int(220*local)))

In [ ]:
# ==========================
# Scene visuals
# ==========================
def current_scene(t: float):
    for s in SCENES:
        if s["start"] <= t < s["end"]:
            local = (t - s["start"]) / (s["end"] - s["start"])
            return s, clamp01(local)
    return SCENES[-1], 1.0


def draw_hook(img: Image.Image, t: float, local: float, scene: dict):
    draw_space(img, t, tint=(1, 2, 8))
    draw_lens_distorted_stars(img, t, strength=0.9)
    draw_black_hole(img, t, scale=1.18, disk=True, ring=True)
    dark_gradient_overlay(img, 80, 195)
    draw_header(img, scene, local, warm=True)
    draw_big_title(img, scene["title"], int(H * 0.215), warm=True)
    draw_caption_box(img, scene["caption"], y=int(H * 0.765))

    d = ImageDraw.Draw(img, "RGBA")
    d.text((W * 0.50, H * 0.70), "watch the grid, light, and paths", font=FONT_SMALL, fill=(210, 230, 255, 160), anchor="mm")


def draw_curvature(img: Image.Image, t: float, local: float, scene: dict):
    draw_space(img, t, tint=(1, 4, 13))
    draw_curved_grid(img, t, local, strength=0.82, alpha=145)
    draw_gravity_vectors(img, t, local)
    draw_black_hole(img, t, scale=0.95, disk=False, ring=True)
    dark_gradient_overlay(img, 75, 170)
    draw_header(img, scene, local, warm=False)
    draw_big_title(img, scene["title"], int(H * 0.185), warm=False)
    draw_caption_box(img, scene["caption"], y=int(H * 0.775))

    d = ImageDraw.Draw(img, "RGBA")
    # Label the well
    cx, cy = BH_CENTER
    d.text((cx, cy + W * 0.225), "deeper curve = stronger gravity", font=FONT_SMALL, fill=(160, 220, 255, 190), anchor="mm")


def draw_falling(img: Image.Image, t: float, local: float, scene: dict):
    draw_space(img, t, tint=(2, 2, 10))
    draw_curved_grid(img, t, 1.0, strength=0.72, alpha=85)
    draw_orbit_path(img, t, local)
    draw_black_hole(img, t, scale=1.0, disk=True, ring=True)
    draw_tidal_object(img, t, clamp01((local - 0.30) / 0.70))
    dark_gradient_overlay(img, 85, 182)
    draw_header(img, scene, local, warm=True)
    draw_big_title(img, scene["title"], int(H * 0.17), warm=True)
    draw_caption_box(img, scene["caption"], y=int(H * 0.775))


def draw_horizon(img: Image.Image, t: float, local: float, scene: dict):
    draw_space(img, t, tint=(1, 2, 9))
    draw_curved_grid(img, t, 1.0, strength=0.66, alpha=70)
    draw_black_hole(img, t, scale=1.08, disk=False, ring=True)
    draw_event_horizon_cones(img, t, local)
    dark_gradient_overlay(img, 95, 190)
    draw_header(img, scene, local, warm=True)
    draw_big_title(img, scene["title"], int(H * 0.17), warm=True)

    d = ImageDraw.Draw(img, "RGBA")
    cx, cy = BH_CENTER
    r = BH_RADIUS * 1.08
    d.text((cx, cy - r - W * 0.055), "event horizon", font=FONT_LABEL, fill=(255, 190, 110, 230), anchor="mm")
    d.line([cx, cy - r - W * 0.035, cx, cy - r * 0.90], fill=(255, 190, 110, 160), width=2)
    draw_caption_box(img, scene["caption"], y=int(H * 0.775))


def draw_lensing(img: Image.Image, t: float, local: float, scene: dict):
    draw_space(img, t, tint=(2, 3, 13))
    d = ImageDraw.Draw(img, "RGBA")

    # Bright background source behind the black hole
    source_x = W * (0.86 - 0.03 * math.sin(t * 0.7))
    source_y = H * 0.42
    glow = Image.new("RGBA", img.size, (0, 0, 0, 0))
    gd = ImageDraw.Draw(glow, "RGBA")
    gd.ellipse([source_x-24, source_y-24, source_x+24, source_y+24], fill=(180, 220, 255, 95))
    glow = glow.filter(ImageFilter.GaussianBlur(12))
    img.alpha_composite(glow)
    d.ellipse([source_x-5, source_y-5, source_x+5, source_y+5], fill=(235, 248, 255, 230))
    d.text((source_x, source_y - W*0.07), "background light", font=FONT_SMALL, fill=(190, 225, 255, 170), anchor="mm")

    # Bent rays
    fade = ease_out(local)
    draw_light_ray(img, H * 0.36, -W * 0.23 * fade, t, (110, 220, 255, int(175 * fade)), width=3)
    draw_light_ray(img, H * 0.43, -W * 0.14 * fade, t, (255, 210, 130, int(165 * fade)), width=3)
    draw_light_ray(img, H * 0.53, W * 0.18 * fade, t, (140, 230, 255, int(150 * fade)), width=3)

    draw_black_hole(img, t, scale=1.05, disk=False, ring=True)

    # Einstein-ring style arcs
    cx, cy = BH_CENTER
    r = W * 0.224
    for j in range(4):
        a = int((130 - j * 20) * fade)
        d.arc([cx-r-j*8, cy-r-j*8, cx+r+j*8, cy+r+j*8], start=28, end=152, fill=(115, 220, 255, a), width=3)
        d.arc([cx-r-j*8, cy-r-j*8, cx+r+j*8, cy+r+j*8], start=205, end=330, fill=(255, 190, 105, a), width=3)

    dark_gradient_overlay(img, 90, 180)
    draw_header(img, scene, local, warm=False)
    draw_big_title(img, scene["title"], int(H * 0.17), warm=False)
    draw_caption_box(img, scene["caption"], y=int(H * 0.775))


def draw_outro(img: Image.Image, t: float, local: float, scene: dict):
    draw_space(img, t, tint=(1, 2, 9))
    draw_lens_distorted_stars(img, t, strength=1.0)
    draw_curved_grid(img, t, 1.0, strength=0.72, alpha=70)
    draw_black_hole(img, t, scale=1.14, disk=True, ring=True)
    dark_gradient_overlay(img, 145, 215)
    draw_header(img, scene, local, warm=True)
    draw_big_title(img, scene["title"], int(H * 0.22), warm=True)
    draw_caption_box(img, scene["caption"], y=int(H * 0.745))

    d = ImageDraw.Draw(img, "RGBA")
    d.text((W // 2, H * 0.925), "rendered procedurally with Python", font=FONT_SMALL, fill=(220, 230, 255, 130), anchor="mm")

In [ ]:
# ==========================
# Frame composer
# ==========================
def make_frame(frame_idx: int) -> np.ndarray:
    t = frame_idx / FPS
    scene, local = current_scene(t)
    img = Image.new("RGBA", (W, H), (0, 0, 0, 255))

    if scene["kind"] == "hook":
        draw_hook(img, t, local, scene)
    elif scene["kind"] == "curvature":
        draw_curvature(img, t, local, scene)
    elif scene["kind"] == "falling":
        draw_falling(img, t, local, scene)
    elif scene["kind"] == "horizon":
        draw_horizon(img, t, local, scene)
    elif scene["kind"] == "lensing":
        draw_lensing(img, t, local, scene)
    elif scene["kind"] == "outro":
        draw_outro(img, t, local, scene)

    draw_progress_bar(img, t)
    img = vignette(img, 0.58)

    # Smooth fade in/out
    fade = 1.0
    fade = min(fade, clamp01(t / 0.55))
    fade = min(fade, clamp01((VIDEO_SECONDS - t) / 0.80))
    if fade < 1:
        black = Image.new("RGBA", img.size, (0, 0, 0, int(255 * (1 - fade))))
        img.alpha_composite(black)

    return np.asarray(img.convert("RGB"))

In [ ]:
# ==========================
# Render storyboard preview
# ==========================
preview_times = [1.4, 8.0, 16.8, 25.0, 33.0, 42.0]
frames = [Image.fromarray(make_frame(int(t * FPS))) for t in preview_times]
thumb_w = W // 2
thumb_h = H // 2
contact = Image.new("RGB", (thumb_w * 2, thumb_h * 3), (0, 0, 0))
for idx, frame in enumerate(frames):
    thumb = frame.resize((thumb_w, thumb_h), Image.Resampling.LANCZOS)
    contact.paste(thumb, ((idx % 2) * thumb_w, (idx // 2) * thumb_h))
contact.save(STORYBOARD_NAME, quality=95)
print(f"Saved storyboard preview: {STORYBOARD_NAME}")
contact

In [ ]:
# ==========================
# Render the full vertical video
# ==========================
# Tip: keep DRAFT_MODE=True for testing. Set DRAFT_MODE=False in the config cell for a sharper final export.
writer = imageio.get_writer(
    VIDEO_NAME,
    fps=FPS,
    codec="libx264",
    quality=8 if DRAFT_MODE else 9,
    pixelformat="yuv420p",
    macro_block_size=16,
)

for frame_idx in tqdm(range(NFRAMES), desc="Rendering black hole gravity short"):
    writer.append_data(make_frame(frame_idx))

writer.close()

print(f"Saved video: {VIDEO_NAME}")
print(f"Duration: {VIDEO_SECONDS:.1f}s | FPS: {FPS} | Size: {W}x{H}")

In [ ]:
# Display video inside the notebook, if supported
from IPython.display import Video, display

display(Video(str(VIDEO_NAME), embed=True, html_attributes="controls autoplay loop"))

## Easy customization

Edit only the `SCENES` list in the settings cell to change the script text/timing.

For a sharper final render:

1. Set `DRAFT_MODE = False`
2. Run all cells again
3. Export the MP4 from `black_hole_gravity_output/`

Ideas to customize the video:

- Make the black hole bigger: increase `BH_RADIUS`
- Make lensing stronger: increase the ray bend offsets in `draw_lensing`
- Make the video shorter: reduce `VIDEO_SECONDS` and adjust `SCENES`
- Change the Shorts text: edit each scene’s `eyebrow`, `title`, and `caption`
- Add voiceover later by combining this MP4 with audio in CapCut, Premiere, DaVinci Resolve, or ffmpeg